In [ ]:
from functools import lru_cache
from itertools import zip_longest


def tolist(f):
    def tolist(*a):
        return tuple(f(*a))

    return tolist


@lru_cache(None)
@tolist
def mkindex(d, n):
    if n == 1:
        yield ((0,),)
    else:
        t = 2 ** (d - 1) - 1
        for i in range(max(1, n - min(t, n - 1)), min(t, n - 1) + 1):
            for l in mkindex(d - 1, i):
                for r in mkindex(d - 1, n - i):
                    yield ((),) + tuple(l + tuple(2 ** i + r for r in r) for i, (l, r) in enumerate(zip_longest(l, r, fillvalue=())))


def pakindex(d, n):
    a = mkindex(d, n)
    child = np.fromiter((2 ** i - 1 + a for a in a for i, a in enumerate(a) for a in a), "i").reshape(-1, n)
    parent = []
    t = child
    for _ in range(d - 1):
        t = (t - 1).clip(0) // 2
        parent.append(t)
    parent = np.array([list(set(i)) for i in np.concatenate(parent, 1)], "i")
    return child, parent


from fastai.vision.all import *
from torch.nn import *

torch.cuda.set_device(1)


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zhMin100.H", "h"))
    m = np.array([open("../tmp/news2016zhMin100.txt").read()]).view("U1")

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class View(Module):
    def __init__(s, k):
        super().__init__()
        s.k = k

    def forward(s, x):
        return x.view(*x.shape[:-2], -1, s.k)


class TNN(Sequential):
    def forward(s, x):
        n = int(math.log2(x.size(1)))
        for _ in range(n):
            x = s[0](x)
        x = s[1](x)
        r = []
        for _ in range(n + 3):
            r.append(x)
            x = s[2](x)
        return torch.cat(r, 1)


class MultiCrossEntropyLoss(Module):
    def forward(s, x, y):
        X, x = x[..., :-1], x[..., -1]
        X = X.log_softmax(2)
        x = F.pad(x[:, :-32].sigmoid(), (0, 32))
        softmaxLos = X[torch.arange(len(X), device=X.device)[:, None, None], child, y[:, None]].sum(2).exp_()
        childLos = (1 - x[:, child]).prod(2)
        parentLos = x[:, parent].prod(2)
        return -(softmaxLos * childLos * parentLos).sum(1).clip_(1e-40).log_().mean()

In [ ]:
child, parent = [torch.from_numpy(i).to("cuda", torch.long) for i in pakindex(6, 8)]
m = Learner(
    DataLoaders(Dat(8, 8192, 10), Dat(8, 8192, 1000)),
    Sequential(
        Embedding(684, 32),
        TNN(
            Sequential(View(64), Linear(64, 32, 0), BatchNorm1dFlat(32, track_running_stats=0), ReLU(1)),
            Sequential(Linear(32, 32, 0), BatchNorm1dFlat(32, track_running_stats=0), ReLU(1), Linear(32, 32, 0)),
            Sequential(BatchNorm1dFlat(32, track_running_stats=0), ReLU(1), Linear(32, 64, 0), View(32)),
        ),
        Linear(32, 32, 0),
        BatchNorm1dFlat(32, track_running_stats=0),
        ReLU(1),
        Linear(32, 685),
    ).cuda(),
    MultiCrossEntropyLoss(),
    cbs=[
        ShowGraphCallback,
        SaveModelCallback(fname="model5d", with_opt=1, reset_on_fit=0),
    ],
)
# m.load('model5')
m.lr_find()

In [ ]:
m.fit_one_cycle(30, _, 9, 99, .1, .01)

In [ ]:
m.fine_tune(10, 1e-4)

In [ ]:
def tree(a):
    a = list(a)
    d = int(math.log2(len(a) + 1))
    for i in range(d):
        for j in range(2 ** i):
            print(end="  " * (2 ** (d - i - 1) - 1))
            if j:
                print(end="  ")
                print(end="  " * (2 ** (d - i - 1) - 1))
            print(end=a.pop(0))
        print()


x, y = next(iter(m.dls[0]))
x = m.model(x)[0]
y = y[0]
print(y)
tree(m.dls.m[x[:, :684].argmax(1).cpu()])